### Loading the text data

In [28]:
with open("tiny-shakespeare.txt","r", encoding="utf-8") as f:
    text=f.read()

### Vocabulary

In [29]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
print("vocabulary : ","".join(chars))
print("vocab size : ",vocab_size)

vocabulary :  
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size :  65


### Simple encoder and decoder maping

In [30]:
stoi={ch:i for i,ch in enumerate(chars)} # creating map like stoi["a"]=10
itos={i:ch for i,ch in enumerate(chars)} # creating map like itos[10]="a"

encode=lambda seq: [stoi[ch] for ch in seq] # encoder lambda function to encode the input seq
decode=lambda seqList: "".join([itos[i] for i in seqList]) # decoder lambda function to decode the output seq

print(encode("hello i am swapnil"))
print(decode(encode("hello i am swapnil")))

[46, 43, 50, 50, 53, 1, 47, 1, 39, 51, 1, 57, 61, 39, 54, 52, 47, 50]
hello i am swapnil


In [31]:
import torch
data=torch.tensor(encode(text))
print("len of data : ",len(data))
print("data type of data : ",data.dtype)

len of data :  1115394
data type of data :  torch.int64


## train and validation dataset

In [32]:
n=int(0.8*len(data))
train_data=data[:n]
val_data=data[n:]

print("train dataset size : ",len(train_data))
print("validation daa=taset size : ",len(val_data))

train dataset size :  892315
validation daa=taset size :  223079


### A simple representation of char by char prediction

In [34]:
decode_custom_for_tensor=lambda seqList: "".join([itos[i.item()] for i in seqList])

block_size=10
print(train_data[:block_size+1])

x=train_data[:block_size]
y=train_data[1:block_size+1]

for t in range(block_size):
    context=x[:t+1]
    target=y[t]
    print(f"when input is : {decode_custom_for_tensor(context)} , the target is : {itos[target.item()]}")

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64])
when input is : F , the target is : i
when input is : Fi , the target is : r
when input is : Fir , the target is : s
when input is : Firs , the target is : t
when input is : First , the target is :  
when input is : First  , the target is : C
when input is : First C , the target is : i
when input is : First Ci , the target is : t
when input is : First Cit , the target is : i
when input is : First Citi , the target is : z


In [47]:
torch.manual_seed(42)

batch_size=5 # the number of independent sequences to process parallaly
block_size=8  # the maximum context length for prediction

# function to get a batch of data
def get_batch(split):

    """
        if len(data)=10
        block_size=7
        batch_size=4

        ix=[1,2,0,2] that means , 
        there would be batch_size number of starting indices and the range should be from 0 to (10-7)-1=> 0 - 2
        
        x=[[1:1+block_size],[2:2+block_size],[0:0+block_size],[2:2+block_size]]
        y=[[2:2+block_size],[3:3+block_size],[1:1+block_size],[3:3+block_size]]

    """

    data=train_data if split=="train" else val_data

    ix=torch.randint(len(data)-block_size, (batch_size,)) # 0 to (len(data)-block_size)-1 : batch_size int numbers

    x=torch.stack([data[i:i+block_size] for i in ix]) # x torch shape : (batch_size x block_size)
    y=torch.stack([data[i+1:i+block_size+1] for i in ix]) # y torch shape : (batch_size x block_size)

    return x,y

xb,yb=get_batch("train")
print("input : ")
print(xb)
print(xb.shape)
print("output : ")
print(yb)
print(yb.shape)

print("--------------------------------------------")

for batch in range(batch_size):
    for block in range(block_size):
        context=xb[batch,:block+1]
        target=yb[batch,block]

        print(f"when the input is {context} , then target is {target}")

input : 
tensor([[47, 57, 10,  1, 39, 52, 42,  1],
        [59, 56,  1, 46, 43, 39, 56, 58],
        [32, 46, 39, 58,  1, 39, 50, 61],
        [26, 53, 58, 46, 47, 52, 45,  1],
        [ 1, 58, 46, 43, 47, 56,  1, 39]])
torch.Size([5, 8])
output : 
tensor([[57, 10,  1, 39, 52, 42,  1, 50],
        [56,  1, 46, 43, 39, 56, 58, 57],
        [46, 39, 58,  1, 39, 50, 61, 39],
        [53, 58, 46, 47, 52, 45,  1, 40],
        [58, 46, 43, 47, 56,  1, 39, 45]])
torch.Size([5, 8])
--------------------------------------------
when the input is tensor([47]) , then target is 57
when the input is tensor([47, 57]) , then target is 10
when the input is tensor([47, 57, 10]) , then target is 1
when the input is tensor([47, 57, 10,  1]) , then target is 39
when the input is tensor([47, 57, 10,  1, 39]) , then target is 52
when the input is tensor([47, 57, 10,  1, 39, 52]) , then target is 42
when the input is tensor([47, 57, 10,  1, 39, 52, 42]) , then target is 1
when the input is tensor([47, 57, 10,

### Bigram model

In [63]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(42)

class BiGramLanguageModel(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)

    def forward(self,idx,targets = None):
        """
            here idx => batch_size x block_size
            so logits => for every token => we would look up the embedding table row for that token
            logits => batch_size x block_size x vocab_size

        """
        logits = self.token_embedding_table(idx)

        B,T,C=logits.shape

        """
        B = batch_size
        T = tensor size / block_size
        C = vocab_size

        For classification, PyTorch's F.cross_entropy() expects:

        input  = (number_of_predictions, number_of_classes)
        target = (number_of_predictions)
        """

        if targets is None:
            loss =None
        else:
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss=F.cross_entropy(logits,targets)


        return logits,loss

    def generate(self,idx,max_new_tokens):

        for _ in range(max_new_tokens):
            """
                here idx => batch_size(B) x block_size(T)
            """
            logits,loss=self(idx) # logits => ( batch_size , block_size , vocab_size )
            """
            we would generate based on the last token
            so , our logits will be model's vocabulary scores for the last input token
            like : ( batch_size , vocab_size )
            this indicates the last token row of the embedding

            """
            logits=logits[:,-1,:] # logits => ( batch_size , vocab_size )
            """
                logits.shape[0]=batch_size
                logits.shape[1]=vocab_size
                we wanna apply softmax on the vocab probs , that is dim=1
            """
            probs=F.softmax(logits,dim=1) 

            # Randomly select a token according to the probabilities for each batch
            # why randomly -> for diverse response every time
            idx_next=torch.multinomial(probs,num_samples=1) # idx_next => ( batch_size x 1 )

            idx=torch.cat((idx,idx_next),dim=1) # concate the next token to the input token idx 

        return idx

m=BiGramLanguageModel(vocab_size)
out,loss=m(xb,yb)
print(m.token_embedding_table)
print(out.shape)   
print(loss)  
print(m.generate(idx=torch.zeros(1,1,dtype=torch.long),max_new_tokens=10)[0].tolist())
print(decode(m.generate(idx=torch.zeros(1,1,dtype=torch.long),max_new_tokens=10)[0].tolist()))

Embedding(65, 65)
torch.Size([40, 65])
tensor(4.7608, grad_fn=<NllLossBackward0>)
[0, 53, 3, 6, 55, 4, 21, 35, 55, 35, 4]

D!CjaB?ij&


In [64]:
#optimizer
optimizer=torch.optim.Adam(m.parameters(),lr=1e-3)

### Train the model for some epochs

In [66]:
batch_size=32

for steps in range(10000):
    xb,yb=get_batch("train")
    logits,loss=m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.4286928176879883


In [67]:
print(decode(m.generate(idx=torch.zeros(1,1,dtype=torch.long),max_new_tokens=1000)[0].tolist()))


Gou thanditithasprd
Bu lotweall fren t pade amy xxngot.pece we oesenderencleirjuY:

He,

Lot;
LO:
T:
Whblll sp
MEdisellthy 'So whiovore ket this tout:
Thil, Kindd mmasheve Butode,
Leartourke rs thatid thf shelll;
Haton bhe
LI

Ay heathe Ond t:
Mybur abrrent t
ERIs ICIIs f quirisbe whinery th ERUSot atats? laire mest,
NICod g cll wrle ig s G y t
Who d; wnou agngd t
ARDYiver y arimiss.
IORo'sc, hes E:
Than&X.
IIldof f

DI gonoho nok'd Misenveyor.
bupacoul ovar w,
I th nt th'dery n:
IE:
Jo I tan, bl it s y; as pel nouborole ive lo mars, errt moturg s, d the berou aghavisy fritlome t turserMy e isowishedr Gor mnche p:
An y
Shen iootht reshif tinal As thy.

Neey veseblood'sind.
NERINEMasul ove'thilioisty th in ire,
AD'lou:

AShil e.

K:
Tho but
Mou
HATy:

O:
HNF n tiswhat indshoul
Andethy be ayou myowaren ud d ld sthothan heaso myor d har es alllaly benjut there walyCA: s, ito, s omord yof
Mathisiad t, tothe
Aneceve 'sth trverd Y ire h kerspritade t cknenspengr:
S:
I aimy y.
Four' t d dd! 